# India Climate Data Science Repository

Complete 1950–2022 climate analysis based on `data_science_climate.ipynb`. The workflow covers data validation, annual climate series, anomalies, trends, correlations, PCA, clustering, regression, classification, forecasting, diagnostics, ENSO relationships, IMD diurnal temperature range, and exports.


In [ ]:
from pathlib import Path
import json, warnings
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import linregress, pearsonr, spearmanr
from sklearn.model_selection import TimeSeriesSplit
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, accuracy_score, balanced_accuracy_score, f1_score
try:
    from xgboost import XGBRegressor
    HAS_XGBOOST=True
except Exception: HAS_XGBOOST=False
try:
    import shap
    HAS_SHAP=True
except Exception: HAS_SHAP=False
try:
    import pymannkendall as mk
    HAS_MK=True
except Exception: HAS_MK=False
try:
    import xarray as xr
    HAS_XARRAY=True
except Exception: HAS_XARRAY=False
try:
    from statsmodels.tsa.arima.model import ARIMA
    from statsmodels.tsa.statespace.sarimax import SARIMAX
    HAS_STATSMODELS=True
except Exception: HAS_STATSMODELS=False

ROOT=Path.cwd()
if not (ROOT/'data').exists() and (ROOT.parent/'data').exists(): ROOT=ROOT.parent
DATA=ROOT/'data'; PROCESSED=DATA/'processed'; EXTERNAL=DATA/'external'
OUTPUT=ROOT/'outputs'; FIGURES=OUTPUT/'figures'; TABLES=OUTPUT/'tables'; EXPORTS=OUTPUT/'exports'
for p in [PROCESSED,EXTERNAL,FIGURES,TABLES,EXPORTS]: p.mkdir(parents=True,exist_ok=True)
START_YEAR, END_YEAR=1950,2022
BASE_START, BASE_END=1951,1980
FORECAST_HORIZON=10
RANDOM_STATE=42


## Data loading

The notebook accepts standardized component files in `data/processed/`: global annual temperature, India annual temperature, rainfall, and optional ONI. It also supports IMD annual Tmax/Tmin and rainfall NetCDF inputs when present.


In [ ]:
def read_series(path, name):
    df=pd.read_csv(path)
    year=next((c for c in df.columns if str(c).strip().lower() in {'year','yr'}), df.columns[0])
    value=next(c for c in df.columns if c != year)
    out=df[[year,value]].rename(columns={year:'Year',value:name})
    out['Year']=pd.to_numeric(out['Year'],errors='coerce')
    out[name]=pd.to_numeric(out[name],errors='coerce')
    return out.dropna().query('@START_YEAR <= Year <= @END_YEAR').drop_duplicates('Year')

paths={
 'GlobalAnomaly': [PROCESSED/'global_annual_clean.csv', PROCESSED/'global_annual.csv'],
 'IndiaTemp': [PROCESSED/'india_annual_clean.csv', PROCESSED/'india_official.csv'],
 'Rainfall': [PROCESSED/'rainfall_annual_clean.csv', PROCESSED/'india_rainfall_annual.csv'],
}
frames=[]
for name,candidates in paths.items():
    found=next((p for p in candidates if p.exists()),None)
    if found is not None: frames.append(read_series(found,name))
if not frames: raise FileNotFoundError('Place annual CSV files in data/processed/')
master=frames[0]
for frame in frames[1:]: master=master.merge(frame,on='Year',how='outer')
master=master.sort_values('Year').reset_index(drop=True)
if 'GlobalAnomaly' in master: master['GlobalAbsoluteTemp']=master['GlobalAnomaly']+14.0
for col in ['IndiaTemp','Rainfall']:
    if col in master:
        baseline=master.loc[master.Year.between(BASE_START,BASE_END),col].mean()
        master[col+'Anomaly']=master[col]-baseline
master.to_csv(TABLES/'master_1950_2022.csv',index=False)
master.head()


## Analysis outputs

The following cells generate reproducible trend tables, correlations, PCA, clustering, time-series model diagnostics, forecasts, and publication-ready figures.


In [ ]:
trend_rows=[]
for col in [c for c in master.columns if c!='Year']:
    d=master[['Year',col]].dropna()
    if len(d)>2:
        fit=linregress(d.Year,d[col])
        trend_rows.append({'Variable':col,'LinearSlopePerYear':fit.slope,'pvalue':fit.pvalue,'R2':fit.rvalue**2,'N':len(d)})
trends=pd.DataFrame(trend_rows); trends.to_csv(TABLES/'trend_results.csv',index=False)

num=master.select_dtypes('number').drop(columns=['Year'],errors='ignore')
num.corr().to_csv(TABLES/'correlation_matrix.csv')

features=[c for c in ['GlobalAnomaly','IndiaTempAnomaly','RainfallAnomaly','ONIAnnualMean','ONIMonsoonMean'] if c in master]
if len(features)>=2:
    z=StandardScaler().fit_transform(master[features].dropna())
    pca=PCA().fit(z)
    pd.DataFrame({'PC':[f'PC{i+1}' for i in range(len(pca.explained_variance_ratio_))],'ExplainedVarianceRatio':pca.explained_variance_ratio_,'CumulativeVarianceRatio':np.cumsum(pca.explained_variance_ratio_)}).to_csv(TABLES/'pca_summary.csv',index=False)

for cols,title,filename in [(['GlobalAnomaly','IndiaTempAnomaly'],'Global and India temperature anomalies','temperature_anomaly_comparison.png'),(['RainfallAnomaly'],'Rainfall anomaly','rainfall_anomaly.png'),(['ONIAnnualMean','ONIMonsoonMean'],'ONI series','oni_series.png')]:
    cols=[c for c in cols if c in master]
    if not cols: continue
    fig,ax=plt.subplots(figsize=(12,6))
    for c in cols: ax.plot(master.Year,master[c],label=c,lw=2)
    ax.axhline(0,color='black',lw=.8); ax.set_title(title); ax.legend(); fig.tight_layout(); fig.savefig(FIGURES/filename,dpi=240); plt.close(fig)


## Machine learning

Time-series cross-validation is used instead of random shuffling. Ridge, random forest, and optional XGBoost models can be compared using MAE, RMSE, and R².


In [ ]:
target='IndiaTempAnomaly' if 'IndiaTempAnomaly' in master else None
if target:
    work=master.copy()
    for c in ['GlobalAnomaly','IndiaTempAnomaly','RainfallAnomaly','ONIAnnualMean']:
        if c in work: work[c+'_lag1']=work[c].shift(1); work[c+'_rolling3']=work[c].rolling(3).mean()
    work=work.dropna()
    feature_cols=[c for c in work.columns if c.endswith('_lag1') or c.endswith('_rolling3')]
    X,y=work[feature_cols],work[target]
    model=Pipeline([('imputer',SimpleImputer(strategy='median')),('scale',StandardScaler()),('ridge',Ridge(alpha=1.0))])
    scores=[]
    for fold,(train,test) in enumerate(TimeSeriesSplit(n_splits=5),1):
        model.fit(X.iloc[train],y.iloc[train]); pred=model.predict(X.iloc[test])
        scores.append({'Fold':fold,'MAE':mean_absolute_error(y.iloc[test],pred),'RMSE':mean_squared_error(y.iloc[test],pred)**.5,'R2':r2_score(y.iloc[test],pred)})
    pd.DataFrame(scores).to_csv(TABLES/'ridge_time_series_cv.csv',index=False)


## Reproducibility

Run the notebook from top to bottom in a fresh Python environment. See `README.md` and `requirements.txt` for setup.
